# CW2 Scaffold A: Blockchain Fraud DetectionRare-event classification on the Elliptic Bitcoin dataset.**Module:** FIN306 / FIN510**Instructions:** Run provided cells, complete `# TODO` sections, write a 2,500-word reflective report.**Data:** `data/elliptic/elliptic_labelled.parquet` (Colab: downloaded from GitHub raw URL if missing).

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    confusion_matrix, precision_recall_curve,
)
from pathlib import Path

pd.set_option("display.float_format", "{:.4f}".format)
print("Setup complete.")


## Step 1: Load Elliptic (labelled subset)

In [ ]:
ell_path = Path("data/elliptic/elliptic_labelled.parquet")

if not ell_path.is_file():
    ell_path.parent.mkdir(parents=True, exist_ok=True)
    import urllib.request
    url = (
        "https://github.com/quinfer/financial-data-science/raw/main/"
        "data/elliptic/elliptic_labelled.parquet"
    )
    print("Downloading Elliptic labelled parquet...")
    urllib.request.urlretrieve(url, ell_path)
    print("Done.")

ell = pd.read_parquet(ell_path)
feat_cols = [c for c in ell.columns if c.startswith("feat_")]
X = ell[feat_cols].values
y = (ell["class"] == 1).astype(int).values
ts = ell["time_step"].values

print(f"Transactions:  {len(ell):,}")
print(f"Features:      {len(feat_cols)}")
print(f"Illicit:       {y.sum():,} ({y.mean():.1%})")
print(f"Time steps:    {ts.min()} to {ts.max()}")


### TODO 1: Plot illicit rate by `time_step` (temporal drift).In your report, explain why drift matters for evaluation.

In [ ]:
# TODO 1: your plot here


## Step 2: Baseline (shuffled stratified CV)

In [ ]:
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_shuffled = cross_val_score(pipe_lr, X, y, cv=cv, scoring="roc_auc")
ap_shuffled = cross_val_score(pipe_lr, X, y, cv=cv, scoring="average_precision")
print("Shuffled 5-fold CV (logistic, balanced)")
print(f"  AUC: {auc_shuffled.mean():.3f} +/- {auc_shuffled.std():.3f}")
print(f"  AP:  {ap_shuffled.mean():.3f} +/- {ap_shuffled.std():.3f}")


## Step 3: Walk-forward temporal validation

In [ ]:
boundaries = [1, 10, 20, 30, 40, 50]
wf_results = []
for i in range(len(boundaries) - 2):
    train_mask = (ts >= boundaries[0]) & (ts < boundaries[i + 1])
    test_mask = (ts >= boundaries[i + 1]) & (ts < boundaries[i + 2])
    if train_mask.sum() < 50 or test_mask.sum() < 50:
        continue
    X_tr, y_tr = X[train_mask], y[train_mask]
    X_te, y_te = X[test_mask], y[test_mask]
    pipe_lr.fit(X_tr, y_tr)
    auc_wf = roc_auc_score(y_te, pipe_lr.predict_proba(X_te)[:, 1])
    wf_results.append({
        "window": f"t={boundaries[i+1]}-{boundaries[i+2]-1}",
        "AUC": auc_wf,
        "illicit_rate": y_te.mean(),
    })
    print(f"  {wf_results[-1]['window']}: AUC={auc_wf:.3f} illicit={y_te.mean():.1%}")

mean_wf = np.mean([r["AUC"] for r in wf_results])
print(f"\nWalk-forward AUC (mean): {mean_wf:.3f}")
print(f"Shuffled CV AUC:         {auc_shuffled.mean():.3f}")
print(f"Look-ahead bias gap:     {auc_shuffled.mean() - mean_wf:+.3f}")


### TODO 2: Dual-axis plot of walk-forward AUC and illicit rate per window.Add horizontal line for shuffled CV AUC.

In [ ]:
# TODO 2: your figure here


## Step 4: Cost-sensitive threshold (last WF test window)

In [ ]:
last_train = (ts >= 1) & (ts < 40)
last_test = (ts >= 40) & (ts < 50)
pipe_lr.fit(X[last_train], y[last_train])
y_prob = pipe_lr.predict_proba(X[last_test])[:, 1]
y_te = y[last_test]

cost_fp, cost_fn = 50, 5000
thresholds = np.arange(0.01, 0.95, 0.01)
costs = []
for t in thresholds:
    y_pred = (y_prob >= t).astype(int)
    cm = confusion_matrix(y_te, y_pred)
    tn, fp, fn, tp = cm.ravel()
    costs.append((fp * cost_fp + fn * cost_fn) / len(y_te))
opt_t = thresholds[int(np.argmin(costs))]
print(f"Optimal threshold (base costs): {opt_t:.2f}")


### TODO 3: Repeat cost curves for cost_fn in {1000, 50000}; plot all three.Discuss regulatory implications in your report.

In [ ]:
# TODO 3: your plots here


## Step 5: Model comparison (walk-forward)

In [ ]:
models = {
    "Logistic": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
    ]),
    "Random Forest": Pipeline([
        ("clf", RandomForestClassifier(
            n_estimators=200, max_depth=10, class_weight="balanced",
            random_state=42, n_jobs=-1)),
    ]),
    "Gradient Boosting": Pipeline([
        ("clf", GradientBoostingClassifier(
            n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)),
    ]),
}
for name, pipe in models.items():
    aucs = []
    for i in range(len(boundaries) - 2):
        tr = (ts >= 1) & (ts < boundaries[i + 1])
        te = (ts >= boundaries[i + 1]) & (ts < boundaries[i + 2])
        if tr.sum() < 50 or te.sum() < 50:
            continue
        pipe.fit(X[tr], y[tr])
        aucs.append(roc_auc_score(y[te], pipe.predict_proba(X[te])[:, 1]))
    print(f"{name:18s} mean WF AUC = {np.mean(aucs):.3f}")


### TODO 4: Grouped bar chart by model and window; interpret bias-variance and deployment risk.

In [ ]:
# TODO 4: your chart here


## Step 6 (extension): graph degrees + re-fit

In [ ]:
import networkx as nx

ep = Path("data/elliptic/elliptic_edges_labelled.parquet")
if not ep.is_file():
    import urllib.request
    ep.parent.mkdir(parents=True, exist_ok=True)
    url = (
        "https://github.com/quinfer/financial-data-science/raw/main/"
        "data/elliptic/elliptic_edges_labelled.parquet"
    )
    urllib.request.urlretrieve(url, ep)
edges_df = pd.read_parquet(ep)
G = nx.from_pandas_edgelist(
    edges_df, source="txId1", target="txId2", create_using=nx.DiGraph
)
in_deg = dict(G.in_degree())
out_deg = dict(G.out_degree())
ell = ell.copy()
ell["in_degree"] = ell["txId"].map(in_deg).fillna(0).astype(int)
ell["out_degree"] = ell["txId"].map(out_deg).fillna(0).astype(int)
ell["total_degree"] = ell["in_degree"] + ell["out_degree"]
print("Graph features merged (extension).")


### TODO 5: Stack degree features onto X; re-run walk-forward for your best model.

In [ ]:
# TODO 5: your extension here
